In [1]:
import pandas as pd
import numpy as np
import glob

# 1. قراءة ملف الكوفي تلقائياً
csv_files = [f for f in glob.glob('*.csv') if 'cleaned' not in f]
file_name = 'dirty_cafe_sales.csv' if 'dirty_cafe_sales.csv' in csv_files else csv_files[0]
df = pd.read_csv(file_name)

print("=== 1. فحص البيانات الخام قبل التنظيف (Raw Data Inspection) ===")
print(f"إجمالي عدد الصفوف والأعمدة: {df.shape}")
print("\nعينة من البيانات الخام:")
print(df.head())

# 2. تنظيف القيم المتضاربة والمفقودة (Handling Inconsistent & Missing Values)
# في بيانات الكوفي توجد نصوص مثل 'ERROR' و 'UNKNOWN'، نحولها إلى NaN
df.replace(['ERROR', 'UNKNOWN', 'nan', 'None', ''], np.nan, inplace=True)

print("\nعدد القيم المفقودة والخاطئة في كل عمود قبل التنظيف:")
print(df.isnull().sum())

# تحويل أعمدة الأرقام من نصوص إلى أرقام عشرية
numeric_cols = ['Quantity', 'Price Per Unit', 'Total Spent']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# استخدام NumPy لتعويض القيم المفقودة في الكمية والسعر
qty_median = np.nanmedian(df['Quantity'])
price_mean = np.nanmean(df['Price Per Unit'])

df['Quantity'] = df['Quantity'].fillna(qty_median)
df['Price Per Unit'] = df['Price Per Unit'].fillna(np.round(price_mean, 2))

# إعادة حساب إجمالي المبيعات (Total Spent) بدقة عبر ضرب المصفوفات في NumPy
quantities = df['Quantity'].to_numpy()
unit_prices = df['Price Per Unit'].to_numpy()
calculated_total = np.round(np.multiply(quantities, unit_prices), 2)

df['Total Spent'] = np.where(df['Total Spent'].isnull(), calculated_total, df['Total Spent'])

# تعويض الأعمدة النصية (اسم المنتج، طريقة الدفع، الموقع) بالقيمة الأكثر تكراراً أو Unknown
categorical_cols = ['Item', 'Payment Method', 'Location']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])

# حذف أي صفوف لا تحتوي على تاريخ معاملة أو تعويضها
if 'Transaction Date' in df.columns:
    df['Transaction Date'] = df['Transaction Date'].ffill().bfill()

# تصنيف حجم الطلبات باستخدام NumPy (np.where)
df['Order_Category'] = np.where(df['Total Spent'] >= np.nanpercentile(df['Total Spent'], 75), 'High Value', 'Standard')

print("\n=== 2. التحقق من القيم المفقودة بعد التنظيف ===")
print(df.isnull().sum())

# 3. الملخص الإحصائي واكتشاف الأنماط باستخدام NumPy
total_spent_arr = df['Total Spent'].to_numpy()
qty_arr = df['Quantity'].to_numpy()

print("\n=== 3. الملخص الإحصائي للمبيعات باستخدام NumPy ===")
print(f"إجمالي إيرادات الكوفي (Total Revenue): ${np.sum(total_spent_arr):,.2f}")
print(f"متوسط قيمة الفاتورة (Mean Transaction): ${np.mean(total_spent_arr):,.2f}")
print(f"الوسيط لقيمة الفاتورة (Median Transaction): ${np.median(total_spent_arr):,.2f}")
print(f"الانحراف المعياري للأسعار (Std Deviation): ${np.std(total_spent_arr):,.2f}")
print(f"أعلى فاتورة (Max Spend): ${np.max(total_spent_arr):,.2f}")
print(f"أقل فاتورة (Min Spend): ${np.min(total_spent_arr):,.2f}")
print(f"متوسط عدد القطع في الطلب الواحد: {np.mean(qty_arr):.2f} قطعة")

print("\n=== 4. أعلى المنتجات تحقيقاً للإيرادات (Patterns in Data) ===")
item_summary = df.groupby('Item')['Total Spent'].agg(['count', 'sum', 'mean']).sort_values(by='sum', ascending=False)
print(item_summary)

# 4. حفظ الملف النظيف النهائي
df.to_csv('cleaned_cafe_sales.csv', index=False)
print("\nتم حفظ الملف النظيف بنجاح باسم: cleaned_cafe_sales.csv")

=== 1. فحص البيانات الخام قبل التنظيف (Raw Data Inspection) ===
إجمالي عدد الصفوف والأعمدة: (10000, 8)

عينة من البيانات الخام:
  Transaction ID    Item Quantity Price Per Unit Total Spent  Payment Method  \
0    TXN_1961373  Coffee        2            2.0         4.0     Credit Card   
1    TXN_4977031    Cake        4            3.0        12.0            Cash   
2    TXN_4271903  Cookie        4            1.0       ERROR     Credit Card   
3    TXN_7034554   Salad        2            5.0        10.0         UNKNOWN   
4    TXN_3160411  Coffee        2            2.0         4.0  Digital Wallet   

   Location Transaction Date  
0  Takeaway       2023-09-08  
1  In-store       2023-05-16  
2  In-store       2023-07-19  
3   UNKNOWN       2023-04-27  
4  In-store       2023-06-11  

عدد القيم المفقودة والخاطئة في كل عمود قبل التنظيف:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Lo